# 07 — SHAP Feature Attribution (Honest Out-of-Fold)

**Purpose:** `04_train_classifier.ipynb` reports feature importance via Gini-impurity decrease (`feature_importances.png`), computed from the model's full-data refit. That refit is the same in-sample model object whose accuracy figure (1.00) is known to be optimistic (see `notebooks/AUDIT_FINDINGS.md`). This notebook adds a second, independent feature-attribution method — SHAP — computed honestly: each variant's SHAP values come from a `RandomForestClassifier` fit only on the other four folds, never on that variant's own row. No SHAP value in this notebook was produced by a model that had seen its own label during training.

**Method:** identical 5-fold `StratifiedKFold(random_state=42)` split used in notebooks 04/06. Per fold: fit a fresh `RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)` on the training rows only, then compute `shap.TreeExplainer` SHAP values for the held-out rows. Concatenating the 5 folds' held-out SHAP values gives one honest, out-of-fold SHAP value per variant per feature per class — exactly the same out-of-fold logic notebook 06 used for predictions, applied here to feature attribution instead.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path('data')
MODELS_DIR = Path('models')

FEATURE_COLS = [
    'blosum62_score', 'conservation_score', 'pssm_score', 'position_entropy',
    'heme_distance_angstrom', 'burial_binary', 'is_interface_residue',
    'af2_confidence', 'contact_count_wt',
]

labels_df = pd.read_csv(DATA_DIR / 'training_variants.csv')
seq_df = pd.read_csv(DATA_DIR / 'sequence_features.csv')
struct_df = pd.read_csv(DATA_DIR / 'structural_features.csv')

df = (labels_df[['variant_id', 'gene', 'position', 'severity_label']]
      .merge(seq_df, on='variant_id', how='inner')
      .merge(struct_df, on='variant_id', how='inner'))
df = df.dropna(subset=['heme_distance_angstrom']).reset_index(drop=True)

X = df[FEATURE_COLS].astype(float).values
y_raw = df['severity_label'].values

le = joblib.load(MODELS_DIR / 'label_encoder.pkl')
y = le.transform(y_raw)
classes = list(le.classes_)
print('classes:', classes)
print(f'n={len(df)}')

**Output:**
```
classes: ['benign', 'mild', 'severe']
n=153
```

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
import shap

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

n_samples, n_features = X.shape
n_classes = len(classes)
shap_oof = np.zeros((n_samples, n_features, n_classes))

for fold_i, (train_idx, test_idx) in enumerate(cv.split(X, y)):
    imputer = SimpleImputer(strategy='mean')
    X_train = imputer.fit_transform(X[train_idx])
    X_test = imputer.transform(X[test_idx])

    rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
    rf.fit(X_train, y[train_idx])

    explainer = shap.TreeExplainer(rf)
    sv = np.array(explainer.shap_values(X_test))
    if sv.shape[0] == n_classes:
        sv = np.transpose(sv, (1, 2, 0))  # -> (n_test, n_features, n_classes)
    shap_oof[test_idx] = sv
    print(f'fold {fold_i}: train={len(train_idx)} test={len(test_idx)} shap_shape={sv.shape}')

print('\nshap_oof full shape:', shap_oof.shape)

**Output:**
```
fold 0: train=122 test=31 shap_shape=(31, 9, 3)
fold 1: train=122 test=31 shap_shape=(31, 9, 3)
fold 2: train=122 test=31 shap_shape=(31, 9, 3)
fold 3: train=123 test=30 shap_shape=(30, 9, 3)
fold 4: train=123 test=30 shap_shape=(30, 9, 3)

shap_oof full shape: (153, 9, 3)
```

In [ ]:
mean_abs_per_class = np.mean(np.abs(shap_oof), axis=0)  # (n_features, n_classes)
mean_abs_overall = mean_abs_per_class.mean(axis=1)

importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap_overall': mean_abs_overall,
})
for ci, cls in enumerate(classes):
    importance_df[f'mean_abs_shap_{cls}'] = mean_abs_per_class[:, ci]

importance_df = importance_df.sort_values('mean_abs_shap_overall', ascending=False).reset_index(drop=True)
print('OOF SHAP feature importance (mean |SHAP| across all 153 OOF predictions, averaged over 3 classes):')
print(importance_df.round(4).to_string(index=False))

out = {
    'method': (
        'TreeExplainer SHAP values computed per-fold on a RandomForestClassifier fit only on '
        "that fold's training rows, then evaluated on that fold's held-out rows (5-fold "
        'StratifiedKFold, same split as notebook 04/06). No SHAP value in this file was computed '
        'by a model that had seen its own row during training.'
    ),
    'feature_importance_mean_abs_shap': importance_df.to_dict(orient='records'),
}
with open(DATA_DIR / 'shap_oof_importance.json', 'w') as f:
    json.dump(out, f, indent=2)

np.save(DATA_DIR / 'shap_oof_values.npy', shap_oof)
df[['variant_id', 'gene', 'position', 'severity_label']].to_csv(DATA_DIR / 'shap_oof_variant_order.csv', index=False)
print('\nSaved shap_oof_importance.json, shap_oof_values.npy, shap_oof_variant_order.csv')

**Output:**
```
OOF SHAP feature importance (mean |SHAP| across all 153 OOF predictions, averaged over 3 classes):
               feature  mean_abs_shap_overall  mean_abs_shap_benign  mean_abs_shap_mild  mean_abs_shap_severe
heme_distance_angstrom                 0.0598                0.0632              0.0751                0.0411
    conservation_score                 0.0583                0.0504              0.0685                0.0561
      position_entropy                 0.0551                0.0556              0.0659                0.0438
            pssm_score                 0.0470                0.0386              0.0609                0.0417
        af2_confidence                 0.0430                0.0245              0.0523                0.0523
        blosum62_score                 0.0381                0.0277              0.0495                0.0372
      contact_count_wt                 0.0329                0.0352              0.0414                0.0221
         burial_binary                 0.0102                0.0113              0.0121                0.0073
  is_interface_residue                 0.0027                0.0023              0.0036                0.0022
```

## Figures

`shap_importance_oof_bar.png` (mean |SHAP| bar chart, honest OOF) and `shap_summary_severe_oof.png` (SHAP beeswarm for the 'severe' class) are saved to `notebooks/data/`. Code for both:

```python
import matplotlib.pyplot as plt
import shap
from sklearn.impute import SimpleImputer

X_display = SimpleImputer(strategy='mean').fit_transform(X)  # display only, not used in SHAP computation
severe_idx = classes.index('severe')

plt.figure(figsize=(7, 4.5))
order = importance_df.sort_values('mean_abs_shap_overall')
plt.barh(order['feature'], order['mean_abs_shap_overall'], color='#1D9E75')
plt.xlabel('Mean |SHAP value| (5-fold out-of-fold, averaged over benign/mild/severe)')
plt.title('SHAP feature importance — honest out-of-fold (no leakage)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'shap_importance_oof_bar.png', dpi=150)

shap.summary_plot(shap_oof[:, :, severe_idx], X_display, feature_names=FEATURE_COLS, show=False)
plt.title("SHAP summary — 'severe' class (5-fold out-of-fold)")
plt.tight_layout()
plt.savefig(DATA_DIR / 'shap_summary_severe_oof.png', dpi=150, bbox_inches='tight')
```

## Comparison against the existing Gini-importance figure

`feature_importances.png` (notebook 04) reports mean Gini-impurity decrease from `calibrated_rf`'s full-data refit:

```
heme_distance_angstrom    0.1747
conservation_score        0.1700
position_entropy          0.1658
pssm_score                0.1460
af2_confidence            0.1092
blosum62_score            0.1057
contact_count_wt          0.0997
burial_binary             0.0197
is_interface_residue      0.0091
```

The honest out-of-fold SHAP ranking above is **identical**, feature-for-feature, to this in-sample Gini ranking: `heme_distance_angstrom > conservation_score > position_entropy > pssm_score > af2_confidence > blosum62_score > contact_count_wt > burial_binary > is_interface_residue`. This agreement across two different attribution methods — one computed from an in-sample full-data fit, one computed honestly out-of-fold — is reassuring: it means the model's reliance on heme distance, conservation, and position entropy as its top three features is not an artifact of the leakage or in-sample fitting documented elsewhere in this project (`AUDIT_FINDINGS.md`), but a stable property of what the classifier has actually learned. It does **not** address the separate, unrelated finding that the HBA1 severity-flip errors are concentrated in a gene the model is nominally good at — SHAP attributes how much each feature moved a given prediction, not whether that prediction was correct.

`is_interface_residue` and `burial_binary` are the two weakest features under both methods, despite one of the 8 severity-flip cases (`HBA1-Lys127Asn`) being the only flip with `is_interface_residue=True` — consistent with notebook 06's observation that interface status was doing most of the (incorrect) work in that specific prediction, even though the feature is globally weak. Low global importance does not preclude a feature mattering disproportionately for individual misclassified cases.